In [14]:
import os
from glob import glob
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm

In [8]:
sp500 = np.load("sp500/SP500.npy")
print(sp500.shape)


(474, 2526, 5)


In [5]:
def load_csi_datas(folder_path: str) -> Dict[str, pd.DataFrame]:
    stock = {}
    sample_data = pd.read_csv(f"csi300/SZ000001.csv")

    for _, path in tqdm(enumerate(glob(f"{folder_path}/*.csv"))):
        curr_df = pd.read_csv(path)
        stock_name = os.path.basename(path).split(".")[0]

        if len(curr_df) == len(sample_data):
            stock[stock_name] = curr_df

    if "csi500" in folder_path:
        with open("/Users/abnerteng/git/neural-alpha/keys/use_csi500.lst", "r") as f:
            used_stocks = f.read().splitlines()

        stock = {k: v for k, v in stock.items() if k in used_stocks}

    return stock


def load_alphamix_datas(
    folder_path: str,
    dataset_name: str = "acl18",
    padding: bool = False,
) -> Dict[str, pd.DataFrame]:
    stock = {}

    if dataset_name == "acl18":
        sample_data = pd.read_csv(f"{folder_path}/AAPL.csv")

        for _, path in tqdm(enumerate(glob(f"{folder_path}/*.csv"))):
            curr_df = pd.read_csv(path)
            curr_df.rename(
                columns={v: v.lower() for v in curr_df.columns}, inplace=True
            )
            if padding:
                curr_df = curr_df.ffill().bfill()

            stock_name = os.path.basename(path).split(".")[0]

            if len(curr_df) == len(sample_data):
                stock[stock_name] = curr_df

    elif dataset_name == "sz_50":
        sample_data = pd.read_csv(f"{folder_path}/600000.csv")
        sample_data["Time"] = pd.to_datetime(sample_data["Time"])
        full_date_index = pd.DatetimeIndex(sample_data["Time"].values)

        for _, path in tqdm(enumerate(glob(f"{folder_path}/*.csv"))):
            curr_df = pd.read_csv(path)

            if padding and (len(curr_df) > 5000 and len(curr_df) < len(sample_data)):
                curr_df["Time"] = pd.to_datetime(curr_df["Time"])
                curr_df = curr_df.set_index("Time").reindex(full_date_index)
                curr_df["Date"] = (
                    curr_df["Date"].fillna(method="ffill").fillna(method="bfill")
                )
                curr_df = curr_df.fillna(0)
                curr_df["Date"] = curr_df["Date"].astype(int)

            curr_df.rename(
                columns={v: v.lower() for v in curr_df.columns}, inplace=True
            )
            curr_df["date"] = curr_df["date"].apply(
                lambda x: (
                    f"{str(x)[:4]}-{str(x)[4:6]}-{str(x)[6:]}"
                    if len(str(x)) == 8
                    else str(x)
                )
            )

            stock_name = os.path.basename(path).split(".")[0]

            if len(curr_df) == len(sample_data):
                stock[stock_name] = curr_df

    return stock

In [6]:
csi300_data = load_csi_datas("csi300")
csi500_data = load_csi_datas("csi500")
acl18_data = load_alphamix_datas("acl18", dataset_name="acl18", padding=False)
sz50_data = load_alphamix_datas("sz_50", dataset_name="sz_50", padding=False)
    

836it [00:02, 308.05it/s]
1585it [00:04, 340.40it/s]
87it [00:00, 641.60it/s]
49it [00:00, 78.59it/s]


In [23]:
def no_roll_preprocess_csi(
    data: Dict[str, pd.DataFrame],
    length: int,
    valid_threshold: str = "2020-03-31"
) -> Tuple[Dict[str, pd.DataFrame], ...]:
    array = np.zeros((len(data), length, 5))

    for i, (stock_id, stock_data) in tqdm(enumerate(data.items())):
        if "close" not in stock_data.columns:
            stock_data.rename(
                columns={
                    "datetime": "date",
                    "$open": "open",
                    "$high": "high",
                    "$low": "low",
                    "$close": "close",
                    "$volume": "volume",
                },
                inplace=True,
            )
        train_data = stock_data[stock_data["date"] <= valid_threshold]
        train_feat_max = train_data[["open", "high", "low", "close", "volume"]].max()
        stock_data_copy = stock_data.ffill()
        stock_data_copy["ret"] = stock_data_copy["close"].pct_change().fillna(0)
        stock_data_copy["next_ret"] = stock_data_copy["ret"].shift(-1)
        stock_data_copy["date"] = pd.to_datetime(stock_data_copy["date"])
        normed_data = stock_data_copy[["open", "high", "low", "close", "volume"]].div(
            train_feat_max
        )
        array[i] = np.nan_to_num(
            normed_data.values,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

    return array


def no_roll_preprocess_alphamix(
    data: Dict[str, pd.DataFrame],
    length: int,
    features: List[int],
    dataset: str = "acl18",
) -> Tuple[Dict[str, pd.DataFrame], ...]:
    if dataset == "acl18":
        array = np.zeros((len(data), length, 11))
        feats = [
            "z_open",
            "z_high",
            "z_low",
            "z_close",
            "z_adj close",
            "trend_5",
            "trend_10",
            "trend_15",
            "trend_20",
            "trend_25",
            "trend_30",
        ]
    else:
        array = np.zeros((len(data), length, 10))
        feats = [
            "z_open",
            "z_high",
            "z_low",
            "z_close",
            "trend_5",
            "trend_10",
            "trend_15",
            "trend_20",
            "trend_25",
            "trend_30",
        ]

    for i, (stock_id, stock_data) in tqdm(enumerate(data.items())):
        stock_data_copy = stock_data.ffill()
        stock_data_copy["ret"] = stock_data_copy["close"].pct_change().fillna(0)
        stock_data_copy["next_ret"] = stock_data_copy["ret"].shift(-1)
        stock_data_copy["date"] = pd.to_datetime(stock_data_copy["date"])
        stock_data_copy["z_open"] = (
            stock_data_copy["open"] / stock_data_copy["close"] - 1
        )
        stock_data_copy["z_high"] = (
            stock_data_copy["high"] / stock_data_copy["close"] - 1
        )
        stock_data_copy["z_low"] = stock_data_copy["low"] / stock_data_copy["close"] - 1

        if dataset == "acl18":
            stock_data_copy["z_adj close"] = (
                stock_data_copy["adj close"] / stock_data_copy["close"] - 1
            )

        stock_data_copy["z_close"] = stock_data_copy["close"].pct_change().fillna(0)

        if dataset == "acl18":
            for t in features:
                stock_data_copy[f"trend_{t}"] = (
                    stock_data_copy["adj close"]
                    .rolling(t)
                    .sum()
                    .div(t * stock_data_copy["adj close"])
                    - 1
                )
        else:
            for t in features:
                stock_data_copy[f"trend_{t}"] = (
                    stock_data_copy["close"]
                    .rolling(t)
                    .sum()
                    .div(t * stock_data_copy["close"])
                    - 1
                )
        array[i] = np.nan_to_num(
            stock_data_copy[feats].values,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

    return array



In [24]:
csi300_arr = no_roll_preprocess_csi(csi300_data, length=csi300_data["SZ000001"].shape[0])
csi500_arr = no_roll_preprocess_csi(csi500_data, length=csi300_data["SZ000001"].shape[0])
acl18_arr = no_roll_preprocess_alphamix(
    acl18_data, length=acl18_data["AAPL"].shape[0], features=[5, 10, 15, 20, 25, 30]
)
sz50_arr = no_roll_preprocess_alphamix(
    sz50_data, length=sz50_data["600000"].shape[0], features=[5, 10, 15, 20, 25, 30], dataset="sz_50"
)

482it [00:01, 259.47it/s]
500it [00:01, 275.64it/s]
84it [00:00, 222.82it/s]
26it [00:00, 140.44it/s]


In [25]:
print(csi300_arr.shape, csi500_arr.shape, acl18_arr.shape, sz50_arr.shape)
print(np.isnan(csi300_arr).sum(), np.isnan(csi500_arr).sum(), np.isnan(acl18_arr).sum(), np.isnan(sz50_arr).sum())

(482, 3649, 5) (500, 3649, 5) (84, 1258, 11) (26, 5180, 10)
0 0 0 0


In [36]:
np.save("csi300/csi300.npy", csi300_arr)
np.save("csi500/csi500.npy", csi500_arr)
np.save("acl18/acl18.npy", acl18_arr)
np.save("sz_50/sz_50.npy", sz50_arr)